<a href="https://colab.research.google.com/github/zhouning/alphaearth-training-system/blob/master/colab/paper12_loveda_full_finetune_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

                # Paper 12 LoveDA Full Fine-Tuning Baseline

                This notebook runs the missing LoveDA cross-domain full fine-tuning baseline for Paper 12 on the public LoveDA dataset.

                **Required runtime:** Colab Pro+ A100 40GB. This experiment updates the full Prithvi backbone on 1024x1024 segmentation tiles. L4 is only reasonable for smoke checks after manually reducing batch size.

                **Storage policy:** download LoveDA to Colab local SSD under `/content/AlphaEarth-System/data/weights/raw_data/loveda`, keep checkpoints under `/content/loveda_full_finetune_runs`, and only persist result JSON files to `/content/drive/MyDrive/paper12_results`.

                **Outputs written to Drive:**
                - `loveda_full_finetune_u2r.json`
                - `loveda_full_finetune_r2u.json`
                - `loveda_full_finetune_summary.json`

In [ ]:
# 1. Mount Drive and create the results directory.
                from google.colab import drive
                drive.mount("/content/drive")

                import os

                RESULTS_DIR = "/content/drive/MyDrive/paper12_results"
                os.makedirs(RESULTS_DIR, exist_ok=True)
                print("Drive results directory:", RESULTS_DIR)

In [ ]:
# 2. GPU, Python, and disk sanity check.
                !nvidia-smi
                !python --version
                !df -h /content

In [ ]:
# 3. Clone the public training repo into local SSD.
                %cd /content
                !rm -rf /content/AlphaEarth-System
                !git clone https://github.com/zhouning/alphaearth-training-system.git /content/AlphaEarth-System
                %cd /content/AlphaEarth-System
                !git log --oneline -3

In [ ]:
# 4. Install the local package and notebook-only helpers.
                %cd /content/AlphaEarth-System
                !pip install -q -e . torchgeo pyyaml huggingface_hub

In [ ]:
# 5. Stage Prithvi weights at the path the benchmark expects.
                %cd /content/AlphaEarth-System
                import os
                import shutil
                from huggingface_hub import hf_hub_download

                DRIVE_WEIGHTS = "/content/drive/MyDrive/Prithvi_100M.pt"
                LOCAL_WEIGHTS = "/content/AlphaEarth-System/data/weights/prithvi/Prithvi_100M.pt"
                os.makedirs(os.path.dirname(LOCAL_WEIGHTS), exist_ok=True)

                if os.path.exists(DRIVE_WEIGHTS):
                    shutil.copy(DRIVE_WEIGHTS, LOCAL_WEIGHTS)
                    print("Copied Prithvi weights from Drive.")
                elif not os.path.exists(LOCAL_WEIGHTS):
                    downloaded = hf_hub_download(
                        repo_id="ibm-nasa-geospatial/Prithvi-100M",
                        filename="Prithvi_100M.pt",
                    )
                    shutil.copy(downloaded, LOCAL_WEIGHTS)
                    print("Downloaded Prithvi weights from Hugging Face.")
                else:
                    print("Prithvi weights already present locally.")

                !ls -lh /content/AlphaEarth-System/data/weights/prithvi

In [ ]:
# 6. Download the public LoveDA cache into local SSD and smoke one sample per split.
                %cd /content/AlphaEarth-System
                LOVEDA_ROOT = "/content/AlphaEarth-System/data/weights/raw_data/loveda"
                !python scripts/download_public_datasets.py --dataset loveda --loveda-root data/weights/raw_data/loveda --max-samples 1
                !du -sh /content/AlphaEarth-System/data/weights/raw_data/loveda

In [ ]:
# 7. Dry-run the experiment matrix before launching the full training jobs.
                %cd /content/AlphaEarth-System
                !python -m geoadapter.bench.run_benchmark --config geoadapter/bench/configs/loveda_lulc_full_finetune_u2r.yaml --dry-run
                !python -m geoadapter.bench.run_benchmark --config geoadapter/bench/configs/loveda_lulc_full_finetune_r2u.yaml --dry-run

In [ ]:
# 8. Run the U->R full fine-tuning baseline. Checkpoints stay on local SSD.
                %cd /content/AlphaEarth-System
                !mkdir -p /content/loveda_full_finetune_runs/u2r
                !python -m geoadapter.bench.run_benchmark --config geoadapter/bench/configs/loveda_lulc_full_finetune_u2r.yaml --output /content/drive/MyDrive/paper12_results/loveda_full_finetune_u2r.json --checkpoint-dir /content/loveda_full_finetune_runs/u2r --checkpoint-every 5

In [ ]:
# 9. Run the R->U full fine-tuning baseline. Checkpoints stay on local SSD.
                %cd /content/AlphaEarth-System
                !mkdir -p /content/loveda_full_finetune_runs/r2u
                !python -m geoadapter.bench.run_benchmark --config geoadapter/bench/configs/loveda_lulc_full_finetune_r2u.yaml --output /content/drive/MyDrive/paper12_results/loveda_full_finetune_r2u.json --checkpoint-dir /content/loveda_full_finetune_runs/r2u --checkpoint-every 5

In [ ]:
# 10. Verify result counts, print per-seed mIoU, and persist a compact summary JSON to Drive.
                import json
                from pathlib import Path
                from statistics import mean, stdev

                results_dir = Path("/content/drive/MyDrive/paper12_results")
                u2r_path = results_dir / "loveda_full_finetune_u2r.json"
                r2u_path = results_dir / "loveda_full_finetune_r2u.json"
                summary_path = results_dir / "loveda_full_finetune_summary.json"

                u2r = json.loads(u2r_path.read_text(encoding="utf-8"))
                r2u = json.loads(r2u_path.read_text(encoding="utf-8"))
                expected_rows = 3
                assert len(u2r) == expected_rows, f"expected {expected_rows} U->R rows, got {len(u2r)}"
                assert len(r2u) == expected_rows, f"expected {expected_rows} R->U rows, got {len(r2u)}"

                def metric_summary(rows, metric):
                    values = [float(row[metric]) for row in rows]
                    return {
                        "mean": mean(values),
                        "std": stdev(values) if len(values) > 1 else 0.0,
                        "values": values,
                    }

                for direction, rows in (("U->R", u2r), ("R->U", r2u)):
                    print(direction)
                    for row in rows:
                        print(
                            {
                                "method": row["method"],
                                "seed": row["seed"],
                                "mIoU": round(float(row["mIoU"]), 4),
                            }
                        )

                summary = {
                    "u2r": metric_summary(u2r, "mIoU"),
                    "r2u": metric_summary(r2u, "mIoU"),
                }
                summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
                print("Wrote", summary_path)
                print(json.dumps(summary, indent=2))